# The Generator and the Commentator

In [1]:
from IPython.display import HTML # to display videos
import base64 # to encode videos as base64
from base64 import b64encode # to encode videos as base64
import os # to interact with the operating system
import subprocess # to run commands
import time # to measure execution time
import csv # to save comments
import uuid # to generate unique ids
import cv2 # to split videos
from PIL import Image # to display videos
import pandas as pd # to display comments
import numpy as np # to use Numerical Python
from io import BytesIO #for a binary stream of data in memory

In [2]:
def download(directory, filename):
    # The base URL of the image files in the GitHub repository
    base_url = 'https://raw.githubusercontent.com/Denis2054/RAG-Driven-Generative-AI/main/'

    # Complete URL for the file
    file_url = f"{base_url}{directory}/{filename}"

    # Use curl to download the file, including an Authorization header for the private token
    try:
        # Prepare the curl command
        curl_command = f'curl -o {filename} {file_url}'

        # Execute the curl command
        subprocess.run(curl_command, check=True, shell=True)
        print(f"Downloaded '{filename}' successfully.")
    except subprocess.CalledProcessError:
        print(f"Failed to download '{filename}'. Check the URL, your internet connection and the file path")
     

In [ ]:
def display_video(file_name):
  with open(file_name, 'rb') as file:
      video_data = file.read()

  # Encode the video file as base64
  video_url = b64encode(video_data).decode()

  html = f'''
  <video width="640" height="480" controls>
  '''
  HTML(html)
  return HTML(html)

In [4]:
def split_file(file_name):
  video_path = file_name
  cap = cv2.VideoCapture(video_path)

  frame_number = 0
  while cap.isOpened():
      ret, frame = cap.read()
      if not ret:
          break

      cv2.imwrite(f"frame_{frame_number}.jpg", frame)
      frame_number += 1
      print(f"Frame {frame_number} saved.")

  cap.release()

In [5]:
def generate_comment(response_data):
    try:
        caption = response_data.choices[0].message.content
        return caption
    except (KeyError, AttributeError):
        print("Error extracting caption from response.")
        return "No caption available."

In [6]:
def save_comment(comment, frame_number, file_name):
    path = f"{file_name}.csv"

    write_header = not os.path.exists(path)

    with open(path, 'a', newline='') as f:
        writer = csv.writer(f, delimiter=',', quotechar='"', quoting=csv.QUOTE_MINIMAL)
        if write_header:
            writer.writerow(['ID', 'FrameNumber', 'Comment', 'FileName'])  # Write the header if the file is being created
        unique_id = str(uuid.uuid4())
        writer.writerow([unique_id, frame_number, comment, file_name])


In [7]:
!pip install transformers pillow

In [8]:
import base64
import os
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration

def generate_blip_comments(filename):
    video_folder = "../Ch10/content" 
    total_frames = len([file for file in os.listdir(video_folder) if file.endswith('.jpg')])

    processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

    nb = 3      # sample frequency
    counter = 0 # sample frequency counter
    for frame_number in range(total_frames):
        counter += 1
        if counter == nb and counter < total_frames:
            counter = 0
            print(f"Analyzing frame {frame_number}...")
            image_path = os.path.join(video_folder, f"frame_{frame_number}.jpg")
            try:
                image = Image.open(image_path).convert('RGB')
                inputs = processor(image, return_tensors="pt")
                out = model.generate(**inputs)
                comment = processor.decode(out[0], skip_special_tokens=True)
                save_comment(comment, frame_number, filename)
            except FileNotFoundError:
                print(f"Error: Frame {frame_number} not found.")
            except Exception as e:
                print(f"Unexpected error: {e}")

W0415 19:03:52.656000 151712 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [9]:
def display_comments(file_name):
  path = f"{file_name}.csv"
  df = pd.read_csv(path)
  return df

In [12]:
print("Step 1: Displaying the video")
print("Selecting the video")
file_name = "alpinist1.mp4" 
print(f"Video: {file_name}")

print("2.Downloading video: downloading from GitHub")
directory = "Chapter10/videos"
download(directory,file_name)

print("2.Downloading video: displaying video")
display_video(file_name)

print("Splitting the video")
split_file(file_name)

Step 1: Displaying the video
Selecting the video
Video: alpinist1.mp4
2.Downloading video: downloading from GitHub
Failed to download 'alpinist1.mp4'. Check the URL, your internet connection and the file path
2.Downloading video: displaying video
Splitting the video
Frame 1 saved.
Frame 2 saved.
Frame 3 saved.
Frame 4 saved.
Frame 5 saved.
Frame 6 saved.
Frame 7 saved.
Frame 8 saved.
Frame 9 saved.
Frame 10 saved.
Frame 11 saved.
Frame 12 saved.
Frame 13 saved.
Frame 14 saved.
Frame 15 saved.
Frame 16 saved.
Frame 17 saved.
Frame 18 saved.
Frame 19 saved.
Frame 20 saved.
Frame 21 saved.
Frame 22 saved.
Frame 23 saved.
Frame 24 saved.
Frame 25 saved.
Frame 26 saved.
Frame 27 saved.
Frame 28 saved.
Frame 29 saved.
Frame 30 saved.
Frame 31 saved.
Frame 32 saved.
Frame 33 saved.
Frame 34 saved.
Frame 35 saved.
Frame 36 saved.
Frame 37 saved.
Frame 38 saved.
Frame 39 saved.
Frame 40 saved.
Frame 41 saved.
Frame 42 saved.
Frame 43 saved.
Frame 44 saved.
Frame 45 saved.
Frame 46 saved.
Frame 

In [14]:
print("Commenting video: creating comments")
start_time = time.time()  

generate_blip_comments(file_name)

total_time = time.time() - start_time  

video_folder = "../Ch10/content"  
total_frames = len([file for file in os.listdir(video_folder) if file.endswith('.jpg')])
print(total_frames)

print("Commenting video: displaying comments")
display_comments(file_name)

print(f"Total Time: {total_time:.2f} seconds")  

Commenting video: creating comments


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

c:\Users\dorot\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dorot\.cache\huggingface\hub\models--Salesforce--blip-image-captioning-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Analyzing frame 2...


c:\Users\dorot\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\generation\utils.py:1168: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Analyzing frame 5...
Analyzing frame 8...
Analyzing frame 11...
Analyzing frame 14...
Analyzing frame 17...
Analyzing frame 20...
Analyzing frame 23...
Analyzing frame 26...
Analyzing frame 29...
Analyzing frame 32...
Analyzing frame 35...
Analyzing frame 38...
Analyzing frame 41...
Analyzing frame 44...
Analyzing frame 47...
Analyzing frame 50...
Analyzing frame 53...
Analyzing frame 56...
Analyzing frame 59...
Analyzing frame 62...
Analyzing frame 65...
Analyzing frame 68...
Analyzing frame 71...
Analyzing frame 74...
Analyzing frame 77...
Analyzing frame 80...
Analyzing frame 83...
Analyzing frame 86...
Analyzing frame 89...
Analyzing frame 92...
Analyzing frame 95...
Analyzing frame 98...
Analyzing frame 101...
Analyzing frame 104...
Analyzing frame 107...
Analyzing frame 110...
Analyzing frame 113...
Analyzing frame 116...
Analyzing frame 119...
Analyzing frame 122...
Analyzing frame 125...
Analyzing frame 128...
Analyzing frame 131...
Analyzing frame 134...
Analyzing frame 137...